# Dataset analysis

A summary of the whole raw dataset, independent of any split: every chart the scanner finds under `raw/`, and the usable subset the parser accepts. How a split divides these charts is compared in `01_make_split`.

Plots go to `data/meta/dataset_eda/` and audio durations to `data/meta/audio_durations.csv`; both are committed.

In [ ]:
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

from donderformer.data.chart_stats import NOTE_TYPES, audio_durations, densities, note_counts
from donderformer.data.data_splitter import DataSplitter
from donderformer.data.tja import COURSES
from donderformer.paths import DataPaths

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for the toy dataset.
paths = DataPaths()
COMPUTE_DURATIONS = False    # decode audio for note density; cached, so only new files are decoded
OUT = paths.meta / "dataset_eda"
DURATIONS = paths.meta / "audio_durations.csv"
OUT.mkdir(parents=True, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(OUT / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

def level_key(level):
    return (level == "", float(level) if level != "" else 0)

rows = DataSplitter.scan(paths.raw)
usable = [r for r in rows if not r["reasons"]]
present = [c for c in COURSES if any(r["course"] == c for r in usable)]
print(f"{len({r['tja'] for r in rows})} TJA files, {len(rows)} charts scanned")
print(f"{len(usable)} usable charts from {len({r['audio'] for r in usable})} audio files "
      f"in {len({r['tja'].rsplit('/', 1)[0] for r in usable})} song folders")

## Scan coverage and parser filtering

Charts per course before and after the parser checks, and why charts are rejected. A chart can have several reasons, so the reason counts add up to more than the rejected charts.

In [ ]:
scanned = Counter(r["course"] for r in rows)
kept = Counter(r["course"] for r in usable)
courses = COURSES + tuple(sorted(set(scanned) - set(COURSES)))
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
x = np.arange(len(courses))
axes[0].bar(x - 0.2, [scanned[c] for c in courses], 0.4, label="scanned")
axes[0].bar(x + 0.2, [kept[c] for c in courses], 0.4, label="usable")
axes[0].set_xticks(x, [c or "(none)" for c in courses], rotation=30)
axes[0].set_title("Charts per course")
axes[0].legend()

reasons = Counter(reason.split(":")[0] for r in rows for reason in r["reasons"]).most_common(12)
axes[1].barh([r for r, _ in reasons][::-1], [n for _, n in reasons][::-1])
axes[1].set_title(f"Rejection reasons ({len(rows) - len(usable)} rejected charts)")
save("scan_coverage")

## Courses and star levels (usable charts)

In [ ]:
fig, axes = plt.subplots(1, len(present), figsize=(3.2 * len(present), 3), squeeze=False)
for ax, course in zip(axes[0], present):
    levels = Counter(str(r["level"]) for r in usable if r["course"] == course)
    order = sorted(levels, key=level_key)
    ax.bar([lv or "?" for lv in order], [levels[lv] for lv in order])
    ax.set_title(f"{course} (n={sum(levels.values())})")
    ax.set_xlabel("Level")
axes[0][0].set_ylabel("Charts")
save("levels")

## Genres and BPM

In [ ]:
genres = Counter(r["genre"] for r in usable)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].barh(list(genres)[::-1], list(genres.values())[::-1])
axes[0].set_title("Usable charts per genre")
axes[1].hist([r["bpm"] for r in usable], bins=40)
axes[1].set_xlabel("BPM")
axes[1].set_title("BPM of usable charts")
save("genres_bpm")

## Note types

Share of note events per type, over all usable charts and per course.

In [ ]:
groups = {"all": usable, **{c: [r for r in usable if r["course"] == c] for c in present}}
counts = np.array([note_counts(g) for g in groups.values()], dtype=float)
shares = 100 * counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(9, 3.5))
bottom = np.zeros(len(groups))
for i, label in enumerate(NOTE_TYPES):
    ax.bar(list(groups), shares[:, i], bottom=bottom, label=label)
    bottom += shares[:, i]
ax.set_ylabel("% of note events")
ax.legend(bbox_to_anchor=(1.02, 1))
save("note_types")

## Note density

Playable note events per second of audio (`donderformer.data.chart_stats`). A long note counts once, at its start, and silence counts towards the duration, so this is average density, not peak speed. Set `COMPUTE_DURATIONS = True` once with the audio available; `01_make_split` reuses the same cache.

In [ ]:
durations = audio_durations(paths.raw, {r["audio"] for r in usable}, DURATIONS,
                            compute=COMPUTE_DURATIONS)
density = densities(usable, durations)
print(f"density for {len(density)} / {len(usable)} usable charts")

if density:
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
    bins = np.histogram_bin_edges(list(density.values()), bins=30)
    for course in present:
        values = [density[r["chart_id"]] for r in usable if r["course"] == course and r["chart_id"] in density]
        axes[0].hist(values, bins=bins, histtype="step", weights=np.full(len(values), 100 / max(len(values), 1)),
                     label=f"{course} (n={len(values)})")
        by_level = {}
        for r in usable:
            if r["course"] == course and r["chart_id"] in density and r["level"] != "":
                by_level.setdefault(r["level"], []).append(density[r["chart_id"]])
        levels = sorted(by_level, key=float)
        axes[1].plot(levels, [np.mean(by_level[lv]) for lv in levels], marker="o", label=course)
    axes[0].set_xlabel("Note events / s")
    axes[0].set_ylabel("% of course's charts")
    axes[0].legend()
    axes[1].set_xlabel("Level")
    axes[1].set_ylabel("Mean note events / s")
    axes[1].legend()
    save("density")

## Observations

Written by Pierre from `all_courses_eda_v1` (no filters, so retained = usable), before the parser accepted kusudama `9…9…8` and `#LYRIC`, so counts shift slightly after re-running. His observations about the train/validation/test parts are in `01_make_split`.

The retained dataset contains **1,989 unique audio files and 6,220 charts**. A song can have several charts, one for each difficulty category. All charts sharing an audio file are assigned to the same split, and the overlap check found no shared audio paths between train, validation and test.

### Note density

Density is defined here as **playable note events divided by the full audio duration in seconds**. A long note counts once, at its start. Audio silence is included in the duration. This measures average chart density, rather than peak speed or every aspect of playing difficulty.

Density was computed for **all 6,220 retained charts, with no audio decoding failures**.

Mean density increases across the main categories: **Easy 1.15, Normal 1.73, Hard 2.92, Oni 4.41 and Ura 6.23 events/s**. It also generally increases with star level. For example, across all splits, Oni level 3 averages approximately **2.65 events/s**, compared with **4.79 events/s** at level 8. This supports comparing levels within each category rather than treating all Oni charts as equally dense.

Differences at rare levels should be interpreted with their sample counts. A mean based on one or two charts is not a stable estimate of a whole level. Lines joining the means are visual guides; they do not imply that missing levels were observed.

### Scope and conclusion

The scanner found **12,154 charts**, retained **6,220** and rejected **5,934**. These plots describe the retained subset, not the complete raw dataset. Frequent rejection reasons include BPM changes and nested long notes. A chart can have multiple rejection reasons, so the reason counts cannot be added to obtain the number of rejected charts.